# Train the MLBB tactical network (free, in Google Colab)

Makes `policy.tflite` for the MLBB Research app (**Model** page → *Choose tactical network file*).

It learns **from your own play**: the situation the app saw (HP, enemies, minions, monsters, tower circle, level, minimap...) and what you did (which of the 8 joystick directions or standing still, attack, skills, spell, recall). On the phone it runs in about a millisecond, 5-10 times a second, and its suggestions are mixed into the AI's rules (the safety rules - retreat, tower circle, recall - always come first).

**Before you start**
1. Play a few matches with the app's screen capture on (Coach mode records you whenever you play; more matches = better). A few hours per hero is a good start.
2. App → **Learn** page → *Export my recorded play (CSV)*. The file lands in Downloads/MLBB-Research.
3. *Runtime → Change runtime type*: CPU is enough (the network is tiny).


In [ ]:
# TensorFlow and pandas are already installed in Colab
import numpy as np, pandas as pd, tensorflow as tf, math, json, io
print("TensorFlow", tf.__version__)

## 1. Settings

In [ ]:
HERO = ""            # e.g. "Griffith" to train for one hero only; "" = all your heroes
WATCHED_WEIGHT = 0.5 # moments recorded from heroes you watched (allies while dead, enemies) count this much
EPOCHS = 40
MIN_ROWS = 1500      # the richer 32-number situation is used when at least this many rows have it

## 2. Load your recorded play
Upload one or more `mlbb-my-play-*.csv` files (on a phone, the Google Drive cell below is more reliable).

In [ ]:
# A. Upload from this device
from google.colab import files
up = files.upload()
frames = [pd.read_csv(io.BytesIO(v)) for k, v in up.items() if k.endswith(".csv")]
df = pd.concat(frames, ignore_index=True)
print(len(df), "rows")

In [ ]:
# B. Or read them from Google Drive (put the CSV files in My Drive/MLBB-Research first)
# from google.colab import drive; import glob
# drive.mount("/content/drive")
# df = pd.concat([pd.read_csv(f) for f in glob.glob("/content/drive/MyDrive/MLBB-Research/mlbb-my-play-*.csv")], ignore_index=True)
# print(len(df), "rows")

## 3. Situation → action examples

In [ ]:
def parse(col):
    out = []
    for v in df[col].fillna("").astype(str):
        out.append([float(x) for x in v.split(",")] if v.strip() else None)
    return out

f1 = parse("features")
f2 = parse("features2") if "features2" in df.columns else [None] * len(df)
rich = sum(1 for v in f2 if v is not None and len(v) == 32)
use2 = rich >= MIN_ROWS
N = 32 if use2 else 17
print(f"rows with the 32-number situation: {rich} -> training on {N} inputs")

X, MOVE, CASTS, W = [], [], [], []
for i, row in df.iterrows():
    hero = str(row["hero"])
    if HERO and hero != HERO and not hero.startswith("~"):
        continue
    feat = f2[i] if use2 else f1[i]
    if feat is None or len(feat) != N:
        continue
    mx, my = float(row["move_x"]), float(row["move_y"])
    # 8 joystick directions (0 = right, 2 = down, 4 = left, 6 = up) or 8 = stand still
    move = 8 if math.hypot(mx, my) < 0.4 else int(round(math.atan2(my, mx) / (math.pi / 4))) % 8
    c = int(row["casts"])
    X.append(feat); MOVE.append(move); CASTS.append([(c >> b) & 1 for b in range(6)])
    W.append(WATCHED_WEIGHT if hero.startswith("~") else 1.0)

X = np.array(X, np.float32); MOVE = np.array(MOVE); CASTS = np.array(CASTS, np.float32); W = np.array(W, np.float32)
print(len(X), "examples;", "moves:", np.bincount(MOVE, minlength=9), "casts (attack,s1,s2,ult,spell,recall):", CASTS.sum(0).astype(int))
assert len(X) >= 300, "Too little recorded play: play more matches with screen capture on, then export again."

# last 15% (the most recent play) checks the result
cut = int(len(X) * 0.85)
Xtr, Xva = X[:cut], X[cut:]

## 4. Train

In [ ]:
inp = tf.keras.Input(shape=(N,), name="situation")
h = tf.keras.layers.Dense(128, activation="relu")(inp)
h = tf.keras.layers.Dropout(0.1)(h)
h = tf.keras.layers.Dense(64, activation="relu")(h)
move_out = tf.keras.layers.Dense(9, activation="softmax", name="move")(h)
cast_out = tf.keras.layers.Dense(6, activation="sigmoid", name="casts")(h)
model = tf.keras.Model(inp, [move_out, cast_out])

# rare moves / casts still matter: weight classes by how rare they are
counts = np.bincount(MOVE[:cut], minlength=9) + 1
cw = (len(MOVE[:cut]) / (9 * counts)) ** 0.5
sw_move = W[:cut] * cw[MOVE[:cut]]
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss=["sparse_categorical_crossentropy", "binary_crossentropy"],
              metrics=[["accuracy"], ["binary_accuracy"]])
hist = model.fit(Xtr, [MOVE[:cut], CASTS[:cut]],
                 sample_weight=[sw_move, W[:cut]],
                 validation_data=(Xva, [MOVE[cut:], CASTS[cut:]]),
                 epochs=EPOCHS, batch_size=128, verbose=2,
                 callbacks=[tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)])

## 5. How good is it? (on your most recent play, which it did not train on)

In [ ]:
pm, pc = model.predict(Xva, verbose=0)
acc = (pm.argmax(1) == MOVE[cut:]).mean()
base = (MOVE[cut:] == np.bincount(MOVE[:cut]).argmax()).mean()
print(f"joystick direction right: {acc:.0%}  (always guessing the most common one: {base:.0%})")
names = ["attack", "skill 1", "skill 2", "ultimate", "spell", "recall"]
for b, n in enumerate(names):
    y = CASTS[cut:, b]; p = pc[:, b] > 0.5
    if y.sum() == 0: continue
    prec = (p & (y > 0)).sum() / max(1, p.sum()); rec = (p & (y > 0)).sum() / y.sum()
    print(f"{n:9s} when you pressed it: found {rec:.0%}, and right {prec:.0%} of the times it says press")

## 6. Export to LiteRT (.tflite) for the phone
One output of 15 numbers: 9 joystick choices (they add up to 1) then the 6 press chances.

In [ ]:
export = tf.keras.Model(inp, tf.keras.layers.Concatenate(name="policy")([move_out, cast_out]))
conv = tf.lite.TFLiteConverter.from_keras_model(export)
tfl = conv.convert()
open("/content/policy.tflite", "wb").write(tfl)
json.dump({"inputs": N, "outputs": 15, "examples": int(len(X)), "move_accuracy": float(acc)}, open("/content/policy_info.json", "w"))
print("policy.tflite:", len(tfl) // 1024, "KB")

# quick check: the .tflite gives the same answer as the trained model
it = tf.lite.Interpreter(model_content=tfl); it.allocate_tensors()
it.set_tensor(it.get_input_details()[0]["index"], Xva[:1]); it.invoke()
print("check:", np.allclose(it.get_tensor(it.get_output_details()[0]["index"])[0][:9], pm[0], atol=1e-4))

## 7. Download

In [ ]:
# A. Direct download (works best on a computer)
from google.colab import files
files.download("/content/policy.tflite")

In [ ]:
# B. Save to Google Drive (most reliable on a phone)
from google.colab import drive; import shutil
drive.mount("/content/drive")
shutil.copy("/content/policy.tflite", "/content/drive/MyDrive/policy.tflite")
print("Saved to Google Drive > My Drive: policy.tflite")

## 8. Put it in the app
1. Copy `policy.tflite` to the phone.
2. App → **Model** page → *Choose tactical network file (.tflite)*.
3. The Player page shows *Tactical network: ready* and, while playing, what it suggests and how often it agrees with the rules.
4. Keep playing and re-train every few matches: it gets better with more of your play.
